# Dataset Exploration

## Objective

The goal of this notebook is to understand the structure of the Olist Brazilian E-Commerce dataset before designing the production database and ETL pipeline.

This notebook is intended for exploration only.

No feature engineering, preprocessing, or model training will be performed here.

In [2]:
from pathlib import Path
import pandas as pd

In [3]:
DATA_PATH = Path("../data/raw")

In [4]:
csv_files = sorted(DATA_PATH.glob("*.csv"))

for file in csv_files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [5]:
datasets = {}

for file in csv_files:
    datasets[file.stem] = pd.read_csv(file)

print(f"Loaded {len(datasets)} datasets.")

Loaded 9 datasets.


In [6]:
for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    print(df.shape)
    print(df.head())
    print()

olist_customers_dataset
(99441, 5)
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  

olist_geolocation_dataset
(1000163, 5)
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0         

## Key and Relationship Validation

Before designing the PostgreSQL schema, validate candidate primary keys
and relationships directly from the source data.

In [7]:
customers = datasets["olist_customers_dataset"]
orders = datasets["olist_orders_dataset"]
order_items = datasets["olist_order_items_dataset"]
payments = datasets["olist_order_payments_dataset"]
reviews = datasets["olist_order_reviews_dataset"]
products = datasets["olist_products_dataset"]
sellers = datasets["olist_sellers_dataset"]
geolocation = datasets["olist_geolocation_dataset"]
categories = datasets["product_category_name_translation"]


def check_key(df, columns, table_name):
    """Check whether one or more columns can uniquely identify rows."""

    duplicate_count = df.duplicated(subset=columns).sum()
    null_count = df[columns].isnull().any(axis=1).sum()

    print(f"{table_name}")
    print(f"  Rows: {len(df):,}")
    print(f"  Candidate key: {columns}")
    print(f"  Duplicate keys: {duplicate_count:,}")
    print(f"  Rows with null key values: {null_count:,}")
    print()


check_key(customers, ["customer_id"], "customers")
check_key(orders, ["order_id"], "orders")
check_key(order_items, ["order_id", "order_item_id"], "order_items")
check_key(payments, ["order_id", "payment_sequential"], "payments")
check_key(products, ["product_id"], "products")
check_key(sellers, ["seller_id"], "sellers")
check_key(categories, ["product_category_name"], "product_categories")

customers
  Rows: 99,441
  Candidate key: ['customer_id']
  Duplicate keys: 0
  Rows with null key values: 0

orders
  Rows: 99,441
  Candidate key: ['order_id']
  Duplicate keys: 0
  Rows with null key values: 0

order_items
  Rows: 112,650
  Candidate key: ['order_id', 'order_item_id']
  Duplicate keys: 0
  Rows with null key values: 0

payments
  Rows: 103,886
  Candidate key: ['order_id', 'payment_sequential']
  Duplicate keys: 0
  Rows with null key values: 0

products
  Rows: 32,951
  Candidate key: ['product_id']
  Duplicate keys: 0
  Rows with null key values: 0

sellers
  Rows: 3,095
  Candidate key: ['seller_id']
  Duplicate keys: 0
  Rows with null key values: 0

product_categories
  Rows: 71
  Candidate key: ['product_category_name']
  Duplicate keys: 0
  Rows with null key values: 0



In [8]:
print("CUSTOMER IDENTITY")
print("-" * 50)

print(f"Customer rows: {len(customers):,}")
print(f"Unique customer_id: {customers['customer_id'].nunique():,}")
print(
    f"Unique customer_unique_id: "
    f"{customers['customer_unique_id'].nunique():,}"
)

print()
print(f"Order rows: {len(orders):,}")
print(f"Unique order_id: {orders['order_id'].nunique():,}")
print(f"Unique customer_id in orders: {orders['customer_id'].nunique():,}")

repeat_customers = (
    customers.groupby("customer_unique_id")
    .size()
    .sort_values(ascending=False)
)

print()
print("Orders/customer records for most frequent customer_unique_ids:")
print(repeat_customers.head(10))

CUSTOMER IDENTITY
--------------------------------------------------
Customer rows: 99,441
Unique customer_id: 99,441
Unique customer_unique_id: 96,096

Order rows: 99,441
Unique order_id: 99,441
Unique customer_id in orders: 99,441

Orders/customer records for most frequent customer_unique_ids:
customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
12f5d6e1cbf93dafd9dcc19095df0b3d     6
63cfc61cee11cbe306bff5857d00bfe4     6
dc813062e0fc23409cd255f7f53c7074     6
de34b16117594161a6a89c50b289d35a     6
dtype: int64


In [9]:
def check_foreign_key(
    child_df,
    child_column,
    parent_df,
    parent_column,
    relationship_name,
):
    child_values = set(child_df[child_column].dropna())
    parent_values = set(parent_df[parent_column].dropna())

    unmatched = child_values - parent_values

    print(relationship_name)
    print(f"  Unique child values: {len(child_values):,}")
    print(f"  Unmatched values: {len(unmatched):,}")
    print()


check_foreign_key(
    orders,
    "customer_id",
    customers,
    "customer_id",
    "orders.customer_id -> customers.customer_id",
)

check_foreign_key(
    order_items,
    "order_id",
    orders,
    "order_id",
    "order_items.order_id -> orders.order_id",
)

check_foreign_key(
    order_items,
    "product_id",
    products,
    "product_id",
    "order_items.product_id -> products.product_id",
)

check_foreign_key(
    order_items,
    "seller_id",
    sellers,
    "seller_id",
    "order_items.seller_id -> sellers.seller_id",
)

check_foreign_key(
    payments,
    "order_id",
    orders,
    "order_id",
    "payments.order_id -> orders.order_id",
)

check_foreign_key(
    reviews,
    "order_id",
    orders,
    "order_id",
    "reviews.order_id -> orders.order_id",
)

orders.customer_id -> customers.customer_id
  Unique child values: 99,441
  Unmatched values: 0

order_items.order_id -> orders.order_id
  Unique child values: 98,666
  Unmatched values: 0

order_items.product_id -> products.product_id
  Unique child values: 32,951
  Unmatched values: 0

order_items.seller_id -> sellers.seller_id
  Unique child values: 3,095
  Unmatched values: 0

payments.order_id -> orders.order_id
  Unique child values: 99,440
  Unmatched values: 0

reviews.order_id -> orders.order_id
  Unique child values: 98,673
  Unmatched values: 0



## Core Schema Profiling

Profile null values and categorical ranges before defining PostgreSQL
data types and constraints for the core relational layer.

In [10]:
tables_to_profile = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "order_payments": payments,
    "order_reviews": reviews,
    "products": products,
    "sellers": sellers,
    "product_categories": categories,
}

for table_name, df in tables_to_profile.items():
    print("=" * 70)
    print(table_name.upper())
    print("=" * 70)

    print("\nData types:")
    print(df.dtypes)

    print("\nNull counts:")
    null_counts = df.isnull().sum()
    print(null_counts[null_counts > 0])

    print()

CUSTOMERS

Data types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Null counts:
Series([], dtype: int64)

ORDERS

Data types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Null counts:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

ORDER_ITEMS

Data types:
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

Null counts:
Series([], 

In [11]:
print("ORDER STATUS")
print(orders["order_status"].value_counts(dropna=False))
print()

print("REVIEW SCORE")
print(reviews["review_score"].value_counts(dropna=False).sort_index())
print()

print("PAYMENT TYPE")
print(payments["payment_type"].value_counts(dropna=False))
print()

print("PAYMENT INSTALLMENTS RANGE")
print(
    payments["payment_installments"].min(),
    "to",
    payments["payment_installments"].max(),
)
print()

print("PRICE RANGE")
print(
    order_items["price"].min(),
    "to",
    order_items["price"].max(),
)
print()

print("FREIGHT VALUE RANGE")
print(
    order_items["freight_value"].min(),
    "to",
    order_items["freight_value"].max(),
)

ORDER STATUS
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

REVIEW SCORE
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

PAYMENT TYPE
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

PAYMENT INSTALLMENTS RANGE
0 to 24

PRICE RANGE
0.85 to 6735.0

FREIGHT VALUE RANGE
0.0 to 409.68


In [12]:
timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

print("ORDER TIMESTAMP NULLS")
print(orders[timestamp_columns].isnull().sum())

ORDER TIMESTAMP NULLS
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [13]:
print("REVIEW KEY PROFILE")
print(f"Rows: {len(reviews):,}")
print(f"Unique review_id: {reviews['review_id'].nunique():,}")
print(f"Duplicate review_id rows: {reviews['review_id'].duplicated().sum():,}")
print(f"Null review_id: {reviews['review_id'].isna().sum():,}")

print()
print(f"Unique order_id: {reviews['order_id'].nunique():,}")
print(f"Duplicate order_id rows: {reviews['order_id'].duplicated().sum():,}")
print(f"Null order_id: {reviews['order_id'].isna().sum():,}")

print()
print("Composite (review_id, order_id) duplicates:")
print(
    reviews.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

REVIEW KEY PROFILE
Rows: 99,224
Unique review_id: 98,410
Duplicate review_id rows: 814
Null review_id: 0

Unique order_id: 98,673
Duplicate order_id rows: 551
Null order_id: 0

Composite (review_id, order_id) duplicates:
0
